In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow import keras
from keras import layers, Sequential, Input, layers, optimizers, callbacks
from tensorflow.keras.callbacks import EarlyStopping

import pickle
import json


In [2]:
real = pd.read_csv("/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/final_models/paired_data.csv")
synthetic = pd.read_csv("/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/final_models/synthetic_paired_data.csv")

# Keep only the feature columns both share
feature_cols = [c for c in real.columns if c.endswith("_bad") or c.endswith("_good")]

combined = pd.concat([real[feature_cols], synthetic[feature_cols]], ignore_index=True)
combined.to_csv("combined_paired_data.csv", index=False)

print(f"Real:      {len(real)}")
print(f"Synthetic: {len(synthetic)}")
print(f"Combined:  {len(combined)}")

Real:      451
Synthetic: 8565
Combined:  9016


In [3]:
EXERCISE_FEATURES = {
    "lateral raise": [
        "left_arm_raise",
        "right_arm_raise",
        "left_elbow_angle",
        "right_elbow_angle",
        "torso_lean",
        "arm_symmetry",
        "left_wrist_above_shoulder",
        "right_wrist_above_shoulder",
    ],
}

In [4]:
file_path = "/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/final_models/models/maurice_model/combined_paired_data.csv"

In [5]:
def load_paired_data(csv_path):
    df = pd.read_csv(csv_path)

    bad_cols  = [f"{feat}_bad" for feat in EXERCISE_FEATURES["lateral raise"]]
    good_cols = [f"{feat}_good" for feat in EXERCISE_FEATURES["lateral raise"]]

    X = df[bad_cols]
    y = df[good_cols]

    return X, y

In [6]:
X, y = load_paired_data(file_path)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


X shape: (9016, 8)
y shape: (9016, 8)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [8]:
X_bad_scaler = StandardScaler()
y_good_scaler = StandardScaler()

In [9]:
X_train_scaled = X_bad_scaler.fit_transform(X_train)
X_test_scaled = X_bad_scaler.transform(X_test)

In [10]:
y_train_scaled = y_good_scaler.fit_transform(y_train)
y_test_scaled = y_good_scaler.transform(y_test)

In [11]:
input_dim = X_train_scaled.shape[1]
output_dim = y_train_scaled.shape[1]


model = Sequential()
model.add(Input(shape=(input_dim,)))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.Dense(16, activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dense(output_dim, activation='linear'))


In [12]:
model.compile(loss='mse', optimizer='adam', metrics=["mae"])

In [13]:
es = EarlyStopping(monitor='val_loss', patience=20)

model.fit(X_train_scaled, y_train_scaled, validation_data= (X_test_scaled, y_test_scaled), epochs=300, callbacks=[es])

Epoch 1/300
226/226 [==============================] - 3s 7ms/step - loss: 0.4998 - mae: 0.5200 - val_loss: 0.2217 - val_mae: 0.3561
Epoch 2/300
226/226 [==============================] - 1s 3ms/step - loss: 0.2000 - mae: 0.3380 - val_loss: 0.1329 - val_mae: 0.2732
Epoch 3/300
226/226 [==============================] - 1s 3ms/step - loss: 0.1614 - mae: 0.3020 - val_loss: 0.1150 - val_mae: 0.2512
Epoch 4/300
226/226 [==============================] - 1s 3ms/step - loss: 0.1445 - mae: 0.2845 - val_loss: 0.1046 - val_mae: 0.2366
Epoch 5/300
226/226 [==============================] - 1s 3ms/step - loss: 0.1358 - mae: 0.2749 - val_loss: 0.0980 - val_mae: 0.2269
Epoch 6/300
226/226 [==============================] - 1s 3ms/step - loss: 0.1302 - mae: 0.2691 - val_loss: 0.0931 - val_mae: 0.2211
Epoch 7/300
226/226 [==============================] - 1s 3ms/step - loss: 0.1234 - mae: 0.2612 - val_loss: 0.0908 - val_mae: 0.2164
Epoch 8/300
226/226 [==============================] - 2s 10ms/step -

In [14]:
# Get predictions
y_pred_scaled = model.predict(X_test_scaled)

# Per-feature absolute error for each sample
abs_errors = np.abs(y_test_scaled - y_pred_scaled)

# Max error across features for each sample
max_errors = np.max(abs_errors, axis=1)

# Summary
print(f"Max Error — mean across samples: {np.mean(max_errors):.4f}")
print(f"Max Error — worst sample:        {np.max(max_errors):.4f}")
print(f"MAE (for comparison):             {np.mean(abs_errors):.4f}")

57/57 [==============================] - 0s 1ms/step
Max Error — mean across samples: 0.4077
Max Error — worst sample:        2.9421
MAE (for comparison):             0.1548


In [15]:
y_pred_real = y_good_scaler.inverse_transform(y_pred_scaled)
y_test_real = y_good_scaler.inverse_transform(y_test_scaled)

abs_errors_real = np.abs(y_test_real - y_pred_real)
max_errors_real = np.max(abs_errors_real, axis=1)

print(f"Max Error — mean across samples: {np.mean(max_errors_real):.2f} degrees")
print(f"Max Error — worst sample:        {np.max(max_errors_real):.2f} degrees")
print(f"MAE (for comparison):             {np.mean(abs_errors_real):.2f} degrees")

Max Error — mean across samples: 9.47 degrees
Max Error — worst sample:        62.19 degrees
MAE (for comparison):             3.16 degrees


In [16]:
np.argmax(abs_errors_real, axis=1)

array([3, 0, 3, ..., 2, 2, 0])

In [17]:
os.makedirs("saved_models", exist_ok=True)

In [18]:
def save_model_artifacts(model, X_scaler, y_scaler,
                          exercise_name, output_dir):
    safe_name = exercise_name.replace(" ", "_")

    # Save model
    model.save(os.path.join(output_dir, f"{safe_name}_supervised.keras"))

    # Save both scalers
    with open(os.path.join(output_dir, f"{safe_name}_X_scaler.pkl"), "wb") as f:
        pickle.dump(X_scaler, f)

    with open(os.path.join(output_dir, f"{safe_name}_y_scaler.pkl"), "wb") as f:
        pickle.dump(y_scaler, f)

    # Save metadata
    meta = {
        "exercise":      exercise_name,
        "input_features":  [f"{feat}_bad" for feat in EXERCISE_FEATURES[exercise_name]],
        "output_features": [f"{feat}_good" for feat in EXERCISE_FEATURES[exercise_name]],
        "input_dim":     len(EXERCISE_FEATURES[exercise_name]),
        "output_dim":    len(EXERCISE_FEATURES[exercise_name]),
    }
    with open(os.path.join(output_dir, f"{safe_name}_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\nSaved to: {output_dir}/")
    print(f"  {safe_name}_supervised.keras")
    print(f"  {safe_name}_X_scaler.pkl")
    print(f"  {safe_name}_y_scaler.pkl")
    print(f"  {safe_name}_meta.json")

In [19]:
save_model_artifacts(model, X_bad_scaler, y_good_scaler, "lateral raise", "saved_models")


Saved to: saved_models/
  lateral_raise_supervised.keras
  lateral_raise_X_scaler.pkl
  lateral_raise_y_scaler.pkl
  lateral_raise_meta.json


In [20]:
worst_idx = np.argmax(max_errors_real)
worst_feature = np.argmax(abs_errors_real[worst_idx])
feature_names = [f.replace("_good", "") for f in EXERCISE_FEATURES["lateral raise"]]

print(f"Worst sample index: {worst_idx}")
print(f"Worst feature: {feature_names[worst_feature]}")
print(f"Predicted: {y_pred_real[worst_idx][worst_feature]:.2f}")
print(f"Actual:    {y_test_real[worst_idx][worst_feature]:.2f}")
print(f"Error:     {abs_errors_real[worst_idx][worst_feature]:.2f} degrees")

Worst sample index: 1729
Worst feature: right_arm_raise
Predicted: 42.64
Actual:    104.83
Error:     62.19 degrees
